<a href="https://colab.research.google.com/github/himanshu-sheetlani/ML-Course/blob/main/yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:

!unzip "/content/drive/MyDrive/ML/Yolo/archive.zip" -d "/content/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset_brain tumor_correct/Train/Images/1461.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1462.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1463.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1464.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1465.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1466.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1467.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1468.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1469.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/147.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1470.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images/1471.png  
  inflating: /content/dataset_brain tumor_correct/Train/Images

In [20]:
### 3. Install packages ###

!git clone https://github.com/autogyro/yolo-V8.git
!cd yolo-V8/ && pip install ultralytics

fatal: destination path 'yolo-V8' already exists and is not an empty directory.


In [21]:
import yaml

# Define the dataset paths and class names
data = {
    'path': '/content/dataset_brain tumor_correct',  # Dataset root directory
    'train': 'Train',  # Train images relative path
    'val': 'Valid',    # Validation images relative path
    'test': 'Test',    # Test images relative path
    'nc': 4,           # Number of classes
    'names': ['glioma', 'meningioma', 'notumor', 'pituitary']  # Class names
}

# Write the data.yaml file
with open('/content/data.yaml', 'w') as f:
    yaml.dump(data, f)

print("data.yaml created successfully at /content/data.yaml")

data.yaml created successfully at /content/data.yaml


In [ ]:
### 4. Train model ###

import os

from ultralytics import YOLO


config_path = '/content/data.yaml'


!yolo task=detect mode=train model=yolov8n.pt data={config_path} epochs=50 imgsz=640 plots=True

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

In [ ]:
# ------------------ Fix for Encoding Issues ------------------
# Sometimes, especially on platforms like Google Colab or Windows,
# you might face locale/encoding issues when running external commands.
# So, you override 'getpreferredencoding' to always return 'UTF-8'.

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"

# Patch the 'locale' module's getpreferredencoding method
locale.getpreferredencoding = getpreferredencoding

# ------------------ Run YOLOv8 Validation ------------------
# This command runs validation (evaluation) of a trained YOLO model.

!yolo task=detect mode=val model=runs/detect/train/weights/best.pt data=/content/data.yaml save_json=True

In [ ]:
from IPython.display import Image, display
import glob

!yolo predict model=/content/runs/detect/train/weights/best.pt \
              source=/content/test/images/y701_jpg.rf.81a4472f77fdc1f31537342ceca340c9.jpg \
              project=/content/runs/detect \
              name=predict \
              exist_ok=True


# Find the path of the predicted image (YOLOv8 saves output in 'runs/detect/predict' by default)
predicted_images = glob.glob('/content/runs/detect/predict/*.jpg')  # or .png if needed

# Display the first predicted image
if predicted_images:
    display(Image(filename=predicted_images[0]))
else:
    print("No predicted images found.")


In [ ]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Paths
test_folder = '/content/test/images'
output_folder = '/content/runs/detect'
output_name = 'predictions'
output_path = os.path.join(output_folder, output_name)

# Ensure output directory exists
os.makedirs(output_path, exist_ok=True)

# Load model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# Select up to 16 test images
test_images = [os.path.join(test_folder, img) for img in os.listdir(test_folder) if img.endswith(('.jpg', '.png'))][:16]

# Run predictions and save
results = []
for img_path in test_images:
    result = model.predict(source=img_path, save=True, save_txt=True, project=output_folder, name=output_name, exist_ok=True)
    results.append(result)

# Gather predicted image paths
predicted_images = [os.path.join(output_path, os.path.basename(img_path)) for img_path in test_images]

# Display images in 4x4 grid
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i, img_path in enumerate(predicted_images):
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(img_path), fontsize=8)
        axes[i].axis('off')
    else:
        axes[i].axis('off')
        axes[i].set_title("Image not found", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
### 5. Download results ###

from google.colab import files


!zip -r /content/runs.zip /content/runs

files.download('/content/runs.zip')